In [7]:
import pandas as pd
import numpy as np
import joblib
import os

model_dir = 'model_components'

full_pipeline = joblib.load(os.path.join(model_dir, 'full_classification_pipeline.pkl'))
label_encoder = joblib.load(os.path.join(model_dir, 'label_encoder.pkl'))


# --- Prediction Function ---
def predict_report(report_data: dict):
    df_input = pd.DataFrame([report_data])

    df_input['text_features'] = (
        df_input.get('report_headline', '') + ' ' +
        df_input.get('report_detail', '')
    )

    feature_cols = ['issue_type', 'governorate', 'city', 'report_category', 'text_features']
    df_input = df_input[feature_cols]

    probs = full_pipeline.predict_proba(df_input)[0]
    class_names = label_encoder.classes_

    # 🔹 Convert to normal float + round
    prob_list = [
        {
            "label": class_names[i],
            "score": round(float(probs[i]), 4)
        }
        for i in range(len(class_names))
    ]

    # 🔹 Sort descending
    prob_list = sorted(prob_list, key=lambda x: x["score"], reverse=True)

    prediction = prob_list[0]["label"]
    confidence = prob_list[0]["score"]

    return {
        "prediction": prediction,
        "confidence": confidence,
        "probabilities": prob_list
    }

if __name__ == "__main__":

    incoming_data = {
        "governorate": "Cairo",
        "city": "Nasr City",
        "issue_type": "gas_issue",
        "report_headline": "Gas leak",
        "report_detail": "Strong smell but i don't know where",
        "report_category": "citizen"
    }

    result = predict_report(incoming_data)

    print(result)

{'prediction': 'Unclear', 'confidence': 0.67, 'probabilities': [{'label': 'Unclear', 'score': 0.67}, {'label': 'True', 'score': 0.1934}, {'label': 'Emergency', 'score': 0.1176}, {'label': 'False', 'score': 0.0125}, {'label': 'Repeated', 'score': 0.0036}, {'label': 'Out_of_scope', 'score': 0.0029}]}


In [5]:
import pandas as pd
import numpy as np
import joblib
import os

# --- 1. Load the Unified Pipeline ---
print("🔽 Loading the production pipeline...")
model_dir = 'model_components'
try:
    full_pipeline = joblib.load(os.path.join(model_dir, 'full_classification_pipeline.pkl'))
    label_encoder = joblib.load(os.path.join(model_dir, 'label_encoder.pkl'))
except FileNotFoundError:
    print(f"❌ Error: Model components not found in '{model_dir}'. Please run the training script first.")
    exit()
print("✅ Pipeline and Encoder loaded successfully.")

# --- 2. Prediction Function with Sensitivity Tuning ---
def predict_report(report_data: dict, emergency_threshold=0.3):
    """
    Takes a raw report dictionary, processes it through the pipeline,
    and returns the prediction with custom sensitivity for Emergencies.

    REMOVED from input (no longer needed):
    - report_date, day_of_week, month  → random in synthetic data, no real signal
    - latitude, longitude              → mostly constant values, misleading
    - proximity_to_service             → data leakage (was derived from label)
    - repetition_flag                  → data leakage (was derived from label)
    - text_length                      → derived from label type, not real content
    """
    # 1. Convert dictionary to DataFrame
    df_input = pd.DataFrame([report_data])

    # 2. Feature Engineering — must match training logic exactly
    df_input['text_features'] = (
        df_input['report_headline'].fillna('') + ' ' +
        df_input['report_detail'].fillna('')
    )

    # 3. Keep only the columns the pipeline was trained on
    feature_cols = ['issue_type', 'governorate', 'city', 'report_category', 'text_features']
    df_input = df_input[feature_cols]

    # 4. Get probabilities from the pipeline
    probs       = full_pipeline.predict_proba(df_input)[0]
    class_names = label_encoder.classes_
    prob_dict   = dict(zip(class_names, probs))

    # 5. Apply custom threshold for Emergency (safety-critical class)
    emergency_idx = list(class_names).index('Emergency')
    if prob_dict['Emergency'] >= emergency_threshold:
        final_prediction = 'Emergency'
    else:
        final_prediction = class_names[np.argmax(probs)]

    return final_prediction, prob_dict


# --- 3. Example Usage ---
if __name__ == "__main__":

    # ==========================================================================
    # UPDATED: Removed leakage & misleading fields from the input dict:
    #   ❌ report_date, longitude, latitude  → no real signal
    #   ❌ proximity_to_service              → data leakage
    #   ❌ repetition_flag                   → data leakage
    # ==========================================================================
    new_report = {
        'governorate':      'Sharqia',
        'city':             '10th of Ramadan City',
        'issue_type':       'waste_garbage',
        'report_headline':  '!!! EXPLOSION !!! - Trash Accumulation',
        'report_detail':    'EXPLOSION!! Trash has not been collected from Street 66 for a week.',
        'report_category':  'citizen',
    }

    print("\n--- Input Report ---")
    print(f"Headline : {new_report['report_headline']}")
    print(f"Detail   : {new_report['report_detail']}")

    # Run Prediction
    prediction, all_probs = predict_report(new_report)

    print("\n" + "=" * 40)
    print(f"🔮 PREDICTION: **{prediction}**")
    print("=" * 40)

    print("\nConfidence Scores:")
    sorted_probs = sorted(all_probs.items(), key=lambda x: x[1], reverse=True)
    for label, score in sorted_probs:
        print(f"  - {label:15}: {score:.4f}")

    if prediction == 'Emergency':
        print("\n⚠️  ACTION REQUIRED: Alerting emergency response teams immediately.")

🔽 Loading the production pipeline...
✅ Pipeline and Encoder loaded successfully.

--- Input Report ---
Headline : !!! EXPLOSION !!! - Trash Accumulation
Detail   : EXPLOSION!! Trash has not been collected from Street 66 for a week.

🔮 PREDICTION: **Emergency**

Confidence Scores:
  - Emergency      : 0.9041
  - True           : 0.0537
  - Unclear        : 0.0172
  - False          : 0.0132
  - Repeated       : 0.0072
  - Out_of_scope   : 0.0046

⚠️  ACTION REQUIRED: Alerting emergency response teams immediately.


In [2]:
import pandas as pd
import numpy as np
import joblib
import os

# --- 1. Load the Unified Pipeline ---
print("🔽 Loading the production pipeline...")
model_dir = 'model_components'
try:
    full_pipeline = joblib.load(os.path.join(model_dir, 'full_classification_pipeline.pkl'))
    label_encoder = joblib.load(os.path.join(model_dir, 'label_encoder.pkl'))
except FileNotFoundError:
    print(f"❌ Error: Model components not found in '{model_dir}'. Please run the training script first.")
    exit()
print("✅ Pipeline and Encoder loaded successfully.")

# --- 2. Embedded Test Data (Updated) ---
raw_data = [
    {"governorate":"Suez","city":"El Arbaeen","issue_type":"electricity","report_headline":"[Follow-up] Power Outage","report_detail":"Still not resolved, I reported this last week: Repeated outages every night since last week at Street 27.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Red Sea","city":"Hurghada","issue_type":"road_damage","report_headline":"Road Collapse","report_detail":"Road is flooded and completely impassable near Building 54.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Qalyubia","city":"Shubra El Kheima","issue_type":"water","report_headline":"Contaminated Water","report_detail":"Sewage water is flooding the entrance of Street 92.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Red Sea","city":"Marsa Alam","issue_type":"telecom","report_headline":"Cable Cut","report_detail":"The entire building at Building 30 has no internet since yesterday.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Dakahlia","city":"Mit Ghamr","issue_type":"gas_issue","report_headline":"Gas Smell","report_detail":"Neighbors are evacuating due to the strong gas odor.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Sohag","city":"Sohag City","issue_type":"gas_issue","report_headline":"Valve Damage","report_detail":"A very strong smell of gas detected at Building 52. Everything is broken.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Suez","city":"El Arbaeen","issue_type":"water","report_headline":"Sewage Overflow","report_detail":"The water smells terrible and is brown in color.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Minya","city":"Beni Mazar","issue_type":"water","report_headline":"Sewage Overflow","report_detail":"Water pressure is extremely low, nothing comes from the tap.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Dakahlia","city":"Mansoura","issue_type":"telecom","report_headline":"Phone Line Dead","report_detail":"I heard rumors about the entire building at Building 34 has no internet since yesterday. although my neighbor disagrees.","report_category":"citizen","true_label":"FALSE"},
    {"governorate":"Cairo","city":"Zamalek","issue_type":"road_damage","report_headline":"Road Collapse","report_detail":"A large pothole on Street 93 is damaging vehicles. This is a total mess.","report_category":"business","true_label":"TRUE"},
    {"governorate":"Alexandria","city":"El Mandara","issue_type":"water","report_headline":"Contaminated Water","report_detail":"Main supply line is leaking heavily near the valve. Immediate danger â€” residents are evacuating.","report_category":"business","true_label":"Emergency"},
    {"governorate":"Luxor","city":"Esna","issue_type":"waste_garbage","report_headline":"[Follow-up] Health Hazard Waste","report_detail":"I reported this before but it was not fixed: Mountains of waste blocking the pavement on Street 87.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Cairo","city":"Maadi","issue_type":"telecom","report_headline":"Phone Line Dead","report_detail":"A cable appears to have been cut near Street 85.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Minya","city":"Beni Mazar","issue_type":"waste_garbage","report_headline":"Trash Accumulation","report_detail":"Dead carcass in the middle of the road causing a health risk.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Minya","city":"Minya City","issue_type":"gas_issue","report_headline":"!!! CRITICAL !!! - Pressure Drop","report_detail":"CRITICAL!! The main gas meter is making a loud whistling sound.","report_category":"citizen","true_label":"Emergency"},
    {"governorate":"Sohag","city":"Sohag City","issue_type":"road_damage","report_headline":"[Follow-up] Flooded Road","report_detail":"Still not resolved, I reported this last week: Road is flooded and completely impassable near Building 22.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Sharqia","city":"Zagazig","issue_type":"water","report_headline":"???","report_detail":"...","report_category":"citizen","true_label":"Unclear"},
    {"governorate":"Dakahlia","city":"Mansoura","issue_type":"telecom","report_headline":"No Internet","report_detail":"The entire building at Building 21 has no internet since yesterday. The building is shaking and cracking.","report_category":"business","true_label":"Emergency"},
    {"governorate":"Red Sea","city":"Hurghada","issue_type":"road_damage","report_headline":"[Follow-up] Damaged Pavement","report_detail":"I reported this before but it was not fixed: The pavement is cracked and very dangerous for pedestrians.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Alexandria","city":"El Mandara","issue_type":"waste_garbage","report_headline":"Trash Accumulation","report_detail":"Mountains of waste blocking the pavement on Street 19.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Giza","city":"Mohandessin","issue_type":"gas_issue","report_headline":"Suspected Gas Explosion","report_detail":"Neighbors are evacuating due to the strong gas odor. We are suffering badly.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Luxor","city":"Karnak","issue_type":"telecom","report_headline":"Request for Financial Aid","report_detail":"I need financial assistance, I cannot afford rent this month.","report_category":"citizen","true_label":"Out_of_scope"},
    {"governorate":"Cairo","city":"Nasr City","issue_type":"gas_issue","report_headline":"Pressure Drop","report_detail":"Neighbors are evacuating due to the strong gas odor. Immediate danger â€” residents are evacuating. This is a total mess.","report_category":"citizen","true_label":"Emergency"},
    {"governorate":"Luxor","city":"Karnak","issue_type":"road_damage","report_headline":"Political Complaint","report_detail":"I want to file a complaint about the local council decisions.","report_category":"citizen","true_label":"Out_of_scope"},
    {"governorate":"Red Sea","city":"El Gouna","issue_type":"waste_garbage","report_headline":"[Follow-up] Trash Accumulation","report_detail":"I reported this before but it was not fixed: Trash has not been collected from Street 42 for a week.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Sohag","city":"Akhmim","issue_type":"telecom","report_headline":"!!! FIRE !!! - No Internet","report_detail":"FIRE!! Mobile signal is completely lost in the Street 28 area. It has been like this for two days.","report_category":"citizen","true_label":"Emergency"},
    {"governorate":"Dakahlia","city":"El Gamalia","issue_type":"gas_issue","report_headline":"Noise Complaint","report_detail":"Neighbors are making too much noise late at night at Building 4.","report_category":"business","true_label":"Out_of_scope"},
    {"governorate":"Red Sea","city":"Ras Gharib","issue_type":"electricity","report_headline":"Frequent Voltage Drops","report_detail":"The electricity has been out at Building 78 for hours. It has been like this for two days.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Suez","city":"Ataka","issue_type":"telecom","report_headline":"Political Complaint","report_detail":"I want to file a complaint about the local council decisions.","report_category":"citizen","true_label":"Out_of_scope"},
    {"governorate":"Sharqia","city":"Abu Hammad","issue_type":"gas_issue","report_headline":"[Follow-up] Suspected Gas Explosion","report_detail":"This is the second time I am reporting Neighbors are evacuating due to the strong gas odor.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Qalyubia","city":"Khusus","issue_type":"gas_issue","report_headline":"Pressure Drop","report_detail":"Construction work may have damaged a pipe on Street 83.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Ismailia","city":"Abu Sultan","issue_type":"telecom","report_headline":"Phone Line Dead","report_detail":"The telephone landline has been dead for three days. Please help quickly!","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Qalyubia","city":"Khusus","issue_type":"road_damage","report_headline":"Medical Emergency","report_detail":"Someone collapsed on Street 2, we need an ambulance now.","report_category":"citizen","true_label":"Out_of_scope"},
    {"governorate":"Sohag","city":"Sohag City","issue_type":"water","report_headline":"Contaminated Water","report_detail":"Sewage water is flooding the entrance of Street 136.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Alexandria","city":"El Mandara","issue_type":"waste_garbage","report_headline":"???","report_detail":"Dead carcass in","report_category":"citizen","true_label":"Unclear"},
    {"governorate":"Giza","city":"Mohandessin","issue_type":"road_damage","report_headline":"Large Pothole","report_detail":"Road is flooded and completely impassable near Building 28.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Suez","city":"Suez City","issue_type":"road_damage","report_headline":"Large Pothole","report_detail":"The pavement is cracked and very dangerous for pedestrians. Fix it ASAP, we can't live like this.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Qalyubia","city":"Khusus","issue_type":"road_damage","report_headline":"Damaged Pavement","report_detail":"Someone told me there is road is flooded and completely impassable near Building 50. I could be wrong about this.","report_category":"citizen","true_label":"FALSE"},
    {"governorate":"Red Sea","city":"Hurghada","issue_type":"road_damage","report_headline":"Check this","report_detail":"The pavement is","report_category":"citizen","true_label":"Unclear"},
    {"governorate":"Gharbia","city":"Tanta","issue_type":"waste_garbage","report_headline":"???","report_detail":"Trash has not b","report_category":"citizen","true_label":"Unclear"},
    {"governorate":"Ismailia","city":"Ismailia City","issue_type":"waste_garbage","report_headline":"Dead Animal","report_detail":"Mountains of waste blocking the pavement on Street 66.","report_category":"business","true_label":"TRUE"},
    {"governorate":"Gharbia","city":"El Santa","issue_type":"electricity","report_headline":"Electric Sparking","report_detail":"The electricity has been out at Building 76 for hours.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Dakahlia","city":"El Gamalia","issue_type":"electricity","report_headline":"[Follow-up] Transformer Explosion","report_detail":"Following up on my previous report about A technical failure in the 500kVA transformer near Building 2.","report_category":"business","true_label":"Repeated"},
    {"governorate":"Suez","city":"Ataka","issue_type":"waste_garbage","report_headline":"[Follow-up] Health Hazard Waste","report_detail":"Still not resolved, I reported this last week: Dead carcass in the middle of the road causing a health risk.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Sohag","city":"Akhmim","issue_type":"gas_issue","report_headline":"[Follow-up] Pressure Drop","report_detail":"Resubmitting my complaint about Neighbors are evacuating due to the strong gas odor.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Alexandria","city":"Downtown Alexandria","issue_type":"gas_issue","report_headline":"Suspected Gas Explosion","report_detail":"Low gas pressure, the heater is not working.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Suez","city":"Ain Sokhna","issue_type":"water","report_headline":"Sewage Overflow","report_detail":"The water smells terrible and is brown in color. I see fire and thick smoke. Everything is broken.","report_category":"business","true_label":"Emergency"},
    {"governorate":"Suez","city":"Ain Sokhna","issue_type":"gas_issue","report_headline":"[Follow-up] Valve Damage","report_detail":"Resubmitting my complaint about A very strong smell of gas detected at Building 25.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Gharbia","city":"Mahalla","issue_type":"waste_garbage","report_headline":"Illegal Dumping","report_detail":"Illegal dumping of construction waste near Building 30. We are suffering badly.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Ismailia","city":"El Qantara","issue_type":"water","report_headline":"[Follow-up] No Water Supply","report_detail":"This is the second time I am reporting Main supply line is leaking heavily near the valve.","report_category":"business","true_label":"Repeated"},
    {"governorate":"Giza","city":"Dokki","issue_type":"waste_garbage","report_headline":"Dead Animal","report_detail":"Mountains of waste blocking the pavement on Street 14.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Qalyubia","city":"Khusus","issue_type":"road_damage","report_headline":"[Follow-up] Large Pothole","report_detail":"I reported this before but it was not fixed: A large pothole on Street 110 is damaging vehicles.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Qalyubia","city":"Qalyub","issue_type":"water","report_headline":"[Follow-up] Contaminated Water","report_detail":"This is the second time I am reporting Main supply line is leaking heavily near the valve.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Sohag","city":"Tahta","issue_type":"telecom","report_headline":"Phone Line Dead","report_detail":"I think there might be mobile signal is completely lost in the Street 68 area. although my neighbor disagrees.","report_category":"business","true_label":"FALSE"},
    {"governorate":"Cairo","city":"Maadi","issue_type":"electricity","report_headline":"Construction Permit Question","report_detail":"How do I get a permit to renovate my flat at Building 37?","report_category":"citizen","true_label":"Out_of_scope"},
    {"governorate":"Aswan","city":"Daraw","issue_type":"road_damage","report_headline":"Damaged Pavement","report_detail":"A large pothole on Street 111 is damaging vehicles. Please help quickly!","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Aswan","city":"Daraw","issue_type":"water","report_headline":"No Water Supply","report_detail":"I am not sure but maybe a 4-inch diameter pipe burst in front of Building 56. but the workers said it was already fixed.","report_category":"citizen","true_label":"FALSE"},
    {"governorate":"Sharqia","city":"Belbeis","issue_type":"gas_issue","report_headline":"Construction Permit Question","report_detail":"How do I get a permit to renovate my flat at Building 5?","report_category":"business","true_label":"Out_of_scope"},
    {"governorate":"Sohag","city":"Sohag City","issue_type":"road_damage","report_headline":"Road Collapse","report_detail":"The pavement is cracked and very dangerous for pedestrians. The situation is critical and life-threatening.","report_category":"citizen","true_label":"Emergency"},
    {"governorate":"Aswan","city":"Aswan City","issue_type":"telecom","report_headline":"Cable Cut","report_detail":"The entire building at Building 21 has no internet since yesterday.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Aswan","city":"Edfu","issue_type":"water","report_headline":"Water Leak","report_detail":"Main supply line is leaking heavily near the valve.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Gharbia","city":"El Santa","issue_type":"electricity","report_headline":"Broken Street Light","report_detail":"Power surges are damaging appliances in the whole building.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Luxor","city":"Esna","issue_type":"gas_issue","report_headline":"!!! FIRE !!! - Suspected Gas Explosion","report_detail":"FIRE!! Construction work may have damaged a pipe on Street 49. I already called before and nothing happened.","report_category":"business","true_label":"Emergency"},
    {"governorate":"Alexandria","city":"Downtown Alexandria","issue_type":"telecom","report_headline":"Signal Loss","report_detail":"A friend said he saw the entire building at Building 13 has no internet since yesterday. I could be wrong about this.","report_category":"citizen","true_label":"FALSE"},
    {"governorate":"Giza","city":"6th of October City","issue_type":"water","report_headline":"[Follow-up] Sewage Overflow","report_detail":"Still not resolved, I reported this last week: Water pressure is extremely low, nothing comes from the tap.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Sohag","city":"Sohag City","issue_type":"waste_garbage","report_headline":"Noise Complaint","report_detail":"Neighbors are making too much noise late at night at Building 30.","report_category":"citizen","true_label":"Out_of_scope"},
    {"governorate":"Suez","city":"El Arbaeen","issue_type":"water","report_headline":"Sewage Overflow","report_detail":"Sewage water is flooding the entrance of Street 15.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Red Sea","city":"Ras Gharib","issue_type":"electricity","report_headline":"Electric Sparking","report_detail":"A technical failure in the 500kVA transformer near Building 19.","report_category":"business","true_label":"TRUE"},
    {"governorate":"Giza","city":"6th of October City","issue_type":"water","report_headline":"[Follow-up] Sewage Overflow","report_detail":"This is the second time I am reporting Main supply line is leaking heavily near the valve.","report_category":"business","true_label":"Repeated"},
    {"governorate":"Qalyubia","city":"Khusus","issue_type":"electricity","report_headline":"Medical Emergency","report_detail":"Someone collapsed on Street 76, we need an ambulance now.","report_category":"citizen","true_label":"Out_of_scope"},
    {"governorate":"Suez","city":"Ain Sokhna","issue_type":"road_damage","report_headline":"???","report_detail":"Part of the roa","report_category":"citizen","true_label":"Unclear"},
    {"governorate":"Ismailia","city":"Ismailia City","issue_type":"electricity","report_headline":"Electric Sparking","report_detail":"The electricity has been out at Building 47 for hours.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Luxor","city":"Armant","issue_type":"gas_issue","report_headline":"Valve Damage","report_detail":"A very strong smell of gas detected at Building 38. The building is shaking and cracking. It has been like this for two days.","report_category":"business","true_label":"Emergency"},
    {"governorate":"Giza","city":"6th of October City","issue_type":"gas_issue","report_headline":"Request for Financial Aid","report_detail":"I need financial assistance, I cannot afford rent this month.","report_category":"citizen","true_label":"Out_of_scope"},
    {"governorate":"Red Sea","city":"Marsa Alam","issue_type":"electricity","report_headline":"Broken Street Light","report_detail":"A technical failure in the 500kVA transformer near Building 60. Please help quickly!","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Alexandria","city":"Downtown Alexandria","issue_type":"road_damage","report_headline":"Road Collapse","report_detail":"Road is flooded and completely impassable near Building 54.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Aswan","city":"Kom Ombo","issue_type":"waste_garbage","report_headline":"Illegal Dumping","report_detail":"Trash has not been collected from Street 95 for a week. It has been like this for two days.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Red Sea","city":"Hurghada","issue_type":"telecom","report_headline":"Phone Line Dead","report_detail":"The entire building at Building 73 has no internet since yesterday.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Cairo","city":"Heliopolis","issue_type":"road_damage","report_headline":"Damaged Pavement","report_detail":"Part of the road near Building 70 has completely collapsed. Please help quickly!","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Cairo","city":"Maadi","issue_type":"gas_issue","report_headline":"Pressure Drop","report_detail":"Low gas pressure, the heater is not working.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Luxor","city":"Armant","issue_type":"telecom","report_headline":"Signal Loss","report_detail":"I think there might be mobile signal is completely lost in the Street 16 area. I could be wrong about this.","report_category":"citizen","true_label":"FALSE"},
    {"governorate":"Alexandria","city":"Smouha","issue_type":"road_damage","report_headline":"Damaged Pavement","report_detail":"Road is flooded and completely impassable near Building 71.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Sohag","city":"Akhmim","issue_type":"electricity","report_headline":"Transformer Explosion","report_detail":"I am not sure but maybe a technical failure in the 500kva transformer near Building 44. but I have not seen it myself.","report_category":"citizen","true_label":"FALSE"},
    {"governorate":"Ismailia","city":"El Qantara","issue_type":"water","report_headline":"Burst Pipe","report_detail":"Sewage water is flooding the entrance of Street 24.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Dakahlia","city":"El Gamalia","issue_type":"waste_garbage","report_headline":"[Follow-up] Illegal Dumping","report_detail":"Following up on my previous report about The garbage smell is unbearable and attracting insects.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Qalyubia","city":"Banha","issue_type":"waste_garbage","report_headline":"Health Hazard Waste","report_detail":"Illegal dumping of construction waste near Building 36.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Red Sea","city":"Marsa Alam","issue_type":"electricity","report_headline":"[Follow-up] Broken Street Light","report_detail":"I reported this before but it was not fixed: The electricity has been out at Building 44 for hours.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Suez","city":"Suez City","issue_type":"water","report_headline":"!!! EXPLOSION !!! - No Water Supply","report_detail":"EXPLOSION!! Sewage water is flooding the entrance of Street 97.","report_category":"citizen","true_label":"Emergency"},
    {"governorate":"Cairo","city":"Shubra","issue_type":"road_damage","report_headline":"Flooded Road","report_detail":"Road is flooded and completely impassable near Building 63. We are suffering badly.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Ismailia","city":"Abu Sultan","issue_type":"road_damage","report_headline":"[Follow-up] Road Collapse","report_detail":"I reported this before but it was not fixed: The pavement is cracked and very dangerous for pedestrians.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Cairo","city":"Downtown Cairo","issue_type":"electricity","report_headline":"Electric Sparking","report_detail":"Repeated outages every night since last week at Street 98.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Aswan","city":"Aswan City","issue_type":"electricity","report_headline":"Transformer Explosion","report_detail":"Power surges are damaging appliances in the whole building. Please help quickly!","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Giza","city":"6th of October City","issue_type":"gas_issue","report_headline":"[Follow-up] Gas Leak","report_detail":"This is the second time I am reporting Construction work may have damaged a pipe on Street 103.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Luxor","city":"Luxor City","issue_type":"electricity","report_headline":"Electric Sparking","report_detail":"I think there might be live wires are exposed on the pole near Street 90. though everything looks fine from outside.","report_category":"citizen","true_label":"FALSE"},
    {"governorate":"Giza","city":"Mohandessin","issue_type":"telecom","report_headline":"Signal Loss","report_detail":"I heard rumors about a cable appears to have been cut near Street 119. though everything looks fine from outside.","report_category":"citizen","true_label":"FALSE"},
    {"governorate":"Red Sea","city":"Hurghada","issue_type":"gas_issue","report_headline":"Gas Leak","report_detail":"A very strong smell of gas detected at Building 58. The whole street is blocked.","report_category":"citizen","true_label":"TRUE"},
    {"governorate":"Red Sea","city":"Hurghada","issue_type":"waste_garbage","report_headline":"Health Hazard Waste","report_detail":"Dead carcass in the middle of the road causing a health risk. The building is shaking and cracking.","report_category":"citizen","true_label":"Emergency"},
    {"governorate":"Qalyubia","city":"Khusus","issue_type":"electricity","report_headline":"[Follow-up] Broken Street Light","report_detail":"Still not resolved, I reported this last week: Power surges are damaging appliances in the whole building.","report_category":"citizen","true_label":"Repeated"},
    {"governorate":"Minya","city":"Abu Qurqas","issue_type":"electricity","report_headline":"[Follow-up] Broken Street Light","report_detail":"I reported this before but it was not fixed: Repeated outages every night since last week at Street 99.","report_category":"citizen","true_label":"Repeated"}
]

# --- 3. Prediction Function ---
def predict_report(report_data: dict, emergency_threshold=0.3):
    df_input = pd.DataFrame([report_data])
    df_input['text_features'] = (
        df_input['report_headline'].fillna('') + ' ' +
        df_input['report_detail'].fillna('')
    )
    feature_cols = ['issue_type', 'governorate', 'city', 'report_category', 'text_features']
    df_input = df_input[feature_cols]

    probs       = full_pipeline.predict_proba(df_input)[0]
    class_names = label_encoder.classes_
    prob_dict   = dict(zip(class_names, probs))

    emergency_idx = list(class_names).index('Emergency')
    if prob_dict['Emergency'] >= emergency_threshold:
        final_prediction = 'Emergency'
    else:
        final_prediction = class_names[np.argmax(probs)]

    return final_prediction, prob_dict


# --- 4. Batch Prediction ---
print("\n🔄 Running batch predictions on all test samples...")

results = []
for i, record in enumerate(raw_data):
    true_label  = record.pop('true_label')          # remove before prediction
    prediction, probs = predict_report(record)
    record['true_label'] = true_label                # restore

    results.append({
        'sample_no':        i + 1,
        'headline':         record['report_headline'],
        'true_label':       true_label,
        'predicted_label':  prediction,
        'correct':          true_label.lower() == prediction.lower(),
        'confidence':       round(max(probs.values()), 4),
        **{f"prob_{k}": round(v, 4) for k, v in probs.items()}
    })

results_df = pd.DataFrame(results)

# --- 5. Summary ---
total   = len(results_df)
correct = results_df['correct'].sum()
print(f"\n✅ Accuracy on test samples: {correct}/{total} = {correct/total:.2%}")
print("\n--- Results DataFrame (first 10 rows) ---")
print(results_df[['sample_no', 'headline', 'true_label', 'predicted_label', 'correct', 'confidence']].to_string(index=False))

# --- 6. Save Results ---
results_df.to_csv('prediction_results.csv', index=False)
print("\n💾 Full results saved to 'prediction_results.csv'")

🔽 Loading the production pipeline...
✅ Pipeline and Encoder loaded successfully.

🔄 Running batch predictions on all test samples...

✅ Accuracy on test samples: 97/99 = 97.98%

--- Results DataFrame (first 10 rows) ---
 sample_no                               headline   true_label predicted_label  correct  confidence
         1               [Follow-up] Power Outage     Repeated        Repeated     True      0.9890
         2                          Road Collapse         TRUE            True     True      0.9269
         3                     Contaminated Water         TRUE            True     True      0.7535
         4                              Cable Cut         TRUE            True     True      0.8638
         5                              Gas Smell         TRUE            True     True      0.8481
         6                           Valve Damage         TRUE            True     True      0.8596
         7                        Sewage Overflow         TRUE            True  

In [3]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import os

# ── Load results (generated by inference_batch.py) ──────────────────────────
results_path = 'prediction_results.csv'
if not os.path.exists(results_path):
    print("❌ 'prediction_results.csv' not found. Run inference_batch.py first.")
    exit()

df = pd.read_csv(results_path)

LABELS      = ['Emergency', 'False', 'Out_of_scope', 'Repeated', 'True', 'Unclear']
LABEL_COLORS = {
    'Emergency':    '#E24B4A',
    'False':        '#EF9F27',
    'Out_of_scope': '#7F77DD',
    'Repeated':     '#378ADD',
    'True':         '#1D9E75',
    'Unclear':      '#888780',
}
CORRECT_COLOR   = '#1D9E75'
INCORRECT_COLOR = '#E24B4A'

fig = plt.figure(figsize=(18, 22), facecolor='white')
fig.suptitle('Model Prediction vs True Label — Visual Report', fontsize=18,
             fontweight='bold', y=0.98, color='#2C2C2A')

gs = GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

# ── 1. Sample-by-sample strip: correct / incorrect ───────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.set_title('Sample-by-sample result  (green = correct, red = wrong)', fontsize=13,
              color='#2C2C2A', pad=10)

n = len(df)
colors = [CORRECT_COLOR if c else INCORRECT_COLOR for c in df['correct']]
ax1.bar(df['sample_no'], [1] * n, color=colors, width=0.85, linewidth=0)
ax1.set_xlim(0.5, n + 0.5)
ax1.set_ylim(0, 1.3)
ax1.set_xlabel('Sample number', fontsize=11, color='#5F5E5A')
ax1.set_yticks([])
ax1.spines[['top', 'right', 'left']].set_visible(False)
ax1.spines['bottom'].set_color('#D3D1C7')

correct_patch   = mpatches.Patch(color=CORRECT_COLOR,   label=f'Correct ({df["correct"].sum()})')
incorrect_patch = mpatches.Patch(color=INCORRECT_COLOR, label=f'Incorrect ({(~df["correct"]).sum()})')
ax1.legend(handles=[correct_patch, incorrect_patch], loc='upper right',
           frameon=False, fontsize=11)

acc = df['correct'].mean()
ax1.text(0.01, 1.18, f'Overall accuracy: {acc:.1%}', transform=ax1.transAxes,
         fontsize=12, color='#2C2C2A', fontweight='bold')

# ── 2. Per-class accuracy ─────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_title('Accuracy per class', fontsize=13, color='#2C2C2A', pad=10)

class_acc = (df.groupby('true_label')['correct']
               .agg(['sum', 'count'])
               .rename(columns={'sum': 'correct', 'count': 'total'}))
class_acc['accuracy'] = class_acc['correct'] / class_acc['total']
class_acc = class_acc.reindex([l for l in LABELS if l in class_acc.index])

bar_colors = [LABEL_COLORS.get(l, '#888780') for l in class_acc.index]
bars = ax2.barh(class_acc.index, class_acc['accuracy'], color=bar_colors, height=0.6)
ax2.set_xlim(0, 1.15)
ax2.set_xlabel('Accuracy', fontsize=11, color='#5F5E5A')
ax2.spines[['top', 'right']].set_visible(False)
ax2.spines[['left', 'bottom']].set_color('#D3D1C7')
ax2.tick_params(colors='#5F5E5A')

for bar, (label, row) in zip(bars, class_acc.iterrows()):
    ax2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
             f'{row["accuracy"]:.0%}  ({int(row["correct"])}/{int(row["total"])})',
             va='center', fontsize=10, color='#2C2C2A')

# ── 3. Confusion-style heatmap (true vs predicted) ────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_title('True label vs predicted label', fontsize=13, color='#2C2C2A', pad=10)

present = sorted(df['true_label'].unique())
conf    = pd.crosstab(df['true_label'], df['predicted_label'])
conf    = conf.reindex(index=present, columns=present, fill_value=0)

im = ax3.imshow(conf.values, cmap='YlOrRd', aspect='auto')
ax3.set_xticks(range(len(present)))
ax3.set_yticks(range(len(present)))
ax3.set_xticklabels(present, rotation=35, ha='right', fontsize=10, color='#2C2C2A')
ax3.set_yticklabels(present, fontsize=10, color='#2C2C2A')
ax3.set_xlabel('Predicted', fontsize=11, color='#5F5E5A')
ax3.set_ylabel('True', fontsize=11, color='#5F5E5A')

for i in range(len(present)):
    for j in range(len(present)):
        val = conf.values[i, j]
        if val > 0:
            text_color = 'white' if val > conf.values.max() * 0.6 else '#2C2C2A'
            ax3.text(j, i, str(val), ha='center', va='center',
                     fontsize=11, fontweight='bold', color=text_color)

# ── 4. Distribution: true labels vs predicted labels ─────────────────────────
ax4 = fig.add_subplot(gs[2, :])
ax4.set_title('Label distribution — true vs predicted', fontsize=13, color='#2C2C2A', pad=10)

true_counts = df['true_label'].value_counts().reindex(LABELS, fill_value=0)
pred_counts = df['predicted_label'].value_counts().reindex(LABELS, fill_value=0)

x      = np.arange(len(LABELS))
width  = 0.35

bars_true = ax4.bar(x - width / 2, true_counts.values, width,
                    label='True label', color=[LABEL_COLORS[l] for l in LABELS],
                    alpha=0.85, linewidth=0)
bars_pred = ax4.bar(x + width / 2, pred_counts.values, width,
                    label='Predicted label', color=[LABEL_COLORS[l] for l in LABELS],
                    alpha=0.40, linewidth=1,
                    edgecolor=[LABEL_COLORS[l] for l in LABELS])

ax4.set_xticks(x)
ax4.set_xticklabels(LABELS, fontsize=11, color='#2C2C2A')
ax4.set_ylabel('Count', fontsize=11, color='#5F5E5A')
ax4.spines[['top', 'right']].set_visible(False)
ax4.spines[['left', 'bottom']].set_color('#D3D1C7')
ax4.tick_params(colors='#5F5E5A')

solid_patch  = mpatches.Patch(color='#888780', alpha=0.85, label='True label (solid)')
hollow_patch = mpatches.Patch(color='#888780', alpha=0.40, label='Predicted label (faded)')
ax4.legend(handles=[solid_patch, hollow_patch], frameon=False, fontsize=11)

output_path = 'prediction_comparison_report.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"✅ Visual report saved → '{output_path}'")

✅ Visual report saved → 'prediction_comparison_report.png'
